In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

False

In [2]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['TAVILY_API_KEY'] = userdata.get('TAVILY_API_KEY')

In [4]:
!pip install langchain langchain-core langchain-openai langchain-community langgraph python-dotenv langchain-mcp-adapters langchain-chroma chromadb pypdf typing

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 74.0 MB/s eta 0:00:00
   ━━

In [5]:
from langchain.chat_models import init_chat_model
model = init_chat_model("openai:gpt-5-mini")
model.invoke("Hello world")
print("Environment setup is done")

/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/chat_models.py:431: UserWarning: Unrecognized keys in model profile: ['reasoning_effort_levels']. This may indicate a version mismatch between langchain-core and your provider package. Consider upgrading langchain-core.
  _warn_unknown_profile_keys(self.profile)


Environment setup is done


---
# Part A — Conceptual Questions

## A1. Foundations (Easy)

1. In your own words, explain the sentence: *"An agent is a model calling tools in a loop until
   a task is complete. A harness is everything around that loop."* What specifically counts as
   part of the "harness"?

**Answer:**
An **agent** is the core reasoning loop: the model decides what action to take, the agent calls a tool, reads the result, and repeats until the task is finished.

The **harness** is the complete supporting system around that loop. It includes:

* System prompts and context assembly
* Tool definitions and tool permissions
* State and memory management
* Loop limits, timeouts, retries, and stop conditions
* Input/output validation
* Guardrails and human approvals
* Error handling and fallback logic
* Logging, tracing, monitoring, and cost control
* Authentication, authorization, and checkpointing

In simple terms, the **agent provides intelligence**, while the **harness provides control, safety, reliability, and production readiness**.

**Usecase:**
In a refund agent, the model may decide to check an order and issue a refund. The harness verifies the user, checks refund limits, requests approval if required, prevents duplicate refunds, logs the action, and handles API failures.

**Common Pitfalls:**

* Treating the model as the security layer
* Allowing unlimited tool calls or loops
* Exposing all tools to every user
* Not validating tool inputs and outputs
* Missing tracing, retries, and approval checks

**To Summarize:**
A good one-line answer is:

**“The agent is the reasoning-and-tool-use loop; the harness is the runtime that manages context, state, tools, safety, execution, observability, and lifecycle around that loop.”**



2. Name the four products in the Lang family (LangChain, LangGraph, LangSmith, Deep Agents) and
   state, in one sentence each, what job each one does. Which one is fundamentally different in
   *kind* from the other three, and why?

**Answer:**

* **LangChain:** A high-level open-source framework that provides model, tool, retrieval, and prebuilt agent integrations for quickly building LLM applications and agents.
* **LangGraph:** A low-level orchestration framework and runtime for building stateful, long-running agents with controlled workflows, persistence, branching, and human intervention.
* **LangSmith:** A framework-agnostic engineering platform for tracing, debugging, evaluating, monitoring, and deploying AI applications.
* **Deep Agents:** A batteries-included agent harness built on LangChain and LangGraph, providing planning, filesystem-based context management, subagents, and long-term memory.

**Fundamentally different:**
**LangSmith** is different because it is an engineering and operations **platform**, whereas LangChain, LangGraph, and Deep Agents are open-source building frameworks or harnesses used inside application code.

**Usecase:**
Build a research agent with Deep Agents, customize components through LangChain, control complex execution with LangGraph, and observe and evaluate it using LangSmith.

**Common Pitfalls:**

* Treating LangChain and LangGraph as competitors rather than different abstraction levels.
* Assuming Deep Agents is a new model; it is an agent harness.
* Thinking LangSmith is required to run LangChain applications.

**To Summarize:**
Remember: **LangChain builds quickly, LangGraph orchestrates precisely, Deep Agents provides a ready-made harness, and LangSmith observes and improves the system.**



3. If you found a 2026-dated tutorial using `AgentExecutor` or `initialize_agent`, what would
   you conclude, and what should you use instead?

**Answer:**
A 2026 tutorial using **`AgentExecutor`** or **`initialize_agent`** is likely **outdated** because these APIs belong to LangChain's older agent framework.

Instead, use:

* **LangGraph** for building production-grade, stateful agents.
* **`create_agent()`** in LangChain for simple agent use cases.
* **Deep Agents** when you need a production-ready agent harness with planning, memory, and sub-agents.

**Usecase:**

* **Simple chatbot/agent:** `create_agent()`
* **Production workflows & multi-agent systems:** LangGraph
* **Enterprise-ready agent applications:** Deep Agents

**Common Pitfalls:**

* Learning from outdated tutorials using `AgentExecutor` or `initialize_agent`.
* Using legacy agents for production applications.
* Assuming `AgentExecutor` is the recommended approach in modern LangChain.

**To Summarize:**
**Remember:** `AgentExecutor` and `initialize_agent` are **legacy APIs**. For modern Agentic AI, prefer **LangGraph** (production), **`create_agent()`** (simple agents), or **Deep Agents** (production-ready harness).


4. Why does a `.env` file exist? What specifically goes wrong if you skip it and hardcode an API
   key directly into a notebook cell?

**Answer:**
A **`.env` file** is used to store **sensitive configuration** (API keys, database credentials, secrets) separately from the source code. It allows applications to securely load secrets using environment variables.

If you hardcode an API key in a notebook cell:

* The key can be accidentally pushed to GitHub.
* Anyone with access to the notebook can misuse the key.
* Rotating or changing the key requires modifying the code.
* The same code cannot be easily reused across development, staging, and production environments.

**Usecase:**
Store your OpenAI API key in a `.env` file and load it using `python-dotenv` or `pydantic-settings`, keeping the codebase secure and environment-independent.

**Common Pitfalls:**

* Committing the `.env` file to GitHub.
* Sharing notebooks with hardcoded API keys.
* Using the same API key for all environments.
* Forgetting to add `.env` to `.gitignore`.

**To Summarize:**
A one-line answer:

**"A `.env` file externalizes sensitive configuration from the code, improving security, portability, and maintainability by preventing secrets from being hardcoded or exposed."**


5. What is the difference between a plain text prompt, a message-object list, and a
   dictionary-based message list? Give one situation where each is the natural choice.

**Answer:**

* **Plain Text Prompt:** A single string containing the entire instruction. Best for simple, single-turn interactions.
* **Message-Object List:** A list of typed message objects (e.g., System, Human, AI, Tool) used by frameworks like LangChain. Best for multi-turn conversations and agent workflows.
* **Dictionary-Based Message List:** A list of dictionaries with fields like `role` and `content` (e.g., `{"role": "user", "content": "Hello"}`). This is the standard format used by most LLM APIs such as OpenAI.

**Natural Choice:**

* **Plain Text Prompt:** Simple text generation (e.g., summarize an article).
* **Message-Object List:** Building a LangChain/LangGraph chatbot with memory and tool calling.
* **Dictionary-Based Message List:** Sending chat requests directly to an LLM API.

**Usecase:**
A customer support chatbot uses **message-object lists** internally for orchestration, while the underlying API call often uses a **dictionary-based message list**.

**Common Pitfalls:**

* Using plain text for multi-turn conversations where roles are important.
* Confusing LangChain message objects with API dictionary messages.
* Mixing message formats without proper conversion.

**To Summarize:**
Remember:

* **Plain Text = Simple prompts**
* **Message Objects = Framework abstraction (LangChain/LangGraph)**
* **Dictionary Messages = API communication format**


## A2. Models, Messages, and Templates (Medium)

6. An `AIMessage` carries more than `.content`. Name at least four other fields or attributes it
   can carry, and what each one is actually useful for.


**Answer:**

An `AIMessage` can carry several important attributes besides `.content`:

* **`tool_calls`** – Stores the tools the AI wants to invoke along with their arguments.
* **`usage_metadata`** – Contains token usage (input, output, total) for cost and usage tracking.
* **`response_metadata`** – Holds model-specific details like model name, finish reason, latency, and provider metadata.
* **`id`** – A unique identifier for the message, useful for tracing and debugging.
* **`name`** *(optional)* – Identifies the AI or agent that generated the message, especially in multi-agent systems.

**Usecase:**
In a production agent, `tool_calls` execute external APIs, `usage_metadata` monitors cost, `response_metadata` helps debugging, and `id` enables end-to-end traceability.

**Common Pitfalls:**

* Assuming `.content` contains all useful information.
* Ignoring `tool_calls` when building tool-enabled agents.
* Not tracking `usage_metadata`, leading to poor cost visibility.
* Overlooking `response_metadata` during debugging and monitoring.

**To Summarize:**
Remember the mnemonic **TURI**:

* **T** → `tool_calls`
* **U** → `usage_metadata`
* **R** → `response_metadata`
* **I** → `id`

These are the most commonly used `AIMessage` attributes in production LangChain applications.

7. Explain why streaming produces `AIMessageChunk` objects instead of plain text fragments, and
   what property of these chunks makes them genuinely different from a string split into pieces.


**Answer:**
`AIMessageChunk` objects contain **structured message data**, not just text. Along with partial content, they can carry metadata such as tool calls, function call arguments, response IDs, usage information, and other message attributes.

Unlike plain string fragments, `AIMessageChunk` objects can be **merged together** to reconstruct a complete `AIMessage` while preserving both the generated text and all associated metadata.

**Usecase:**
In a streaming chatbot, each `AIMessageChunk` is received incrementally and merged to build the final response, including any tool calls or structured outputs.

**Common Pitfalls:**

* Treating `AIMessageChunk` as plain text.
* Concatenating only the text and losing metadata (e.g., tool calls or usage info).
* Ignoring chunk merging and processing each chunk independently.

**To Summarize:**
**Key difference:** A **string chunk** contains only text, whereas an **`AIMessageChunk`** is a structured object that carries **text + metadata** and supports **merging into a complete `AIMessage`**.

8. What's the difference between `.batch()` and `.batch_as_completed()`? Describe a real
   situation where you would specifically want the second one over the first.


**Answer:**

* **`.batch()`** executes multiple inputs in parallel and **returns the results in the same order** as the input list, after **all tasks are completed**.
* **`.batch_as_completed()`** executes multiple inputs in parallel but **returns each result as soon as it finishes**, regardless of the original input order.

**Usecase:**
Suppose you're summarizing **1,000 documents** with varying sizes. Using **`.batch_as_completed()`**, you can immediately process or display summaries for smaller documents as they finish instead of waiting for the slowest document to complete.

**Common Pitfalls:**

* Assuming `.batch_as_completed()` preserves input order—it **does not**.
* Using `.batch()` for long-running jobs where early results are needed.
* Forgetting to associate completed results with their original input when using `.batch_as_completed()`.

**To Summarize:**
**Remember:**

* **`.batch()` = Ordered results, wait for all.**
* **`.batch_as_completed()` = Unordered results, stream results as they finish.**
  Use `.batch_as_completed()` when **low latency and incremental processing** are more important than maintaining input order.


9. A `ToolMessage` has both a `.content` field and an `.artifact` field. What's the difference,
   and why would a RAG-style tool specifically want to use `.artifact`?


**Answer:**

* **`.content`** contains the **LLM-readable output** (text) that is sent back to the model for reasoning.
* **`.artifact`** stores the **full structured/raw tool result** (e.g., retrieved documents, metadata, scores, images, JSON) that is **not directly sent to the LLM** but is available to the application.

A **RAG tool** uses **`.artifact`** to preserve complete retrieved documents, metadata, citations, and relevance scores, while sending only the necessary summarized context in `.content`. This reduces token usage and enables citation, debugging, and post-processing.

**Usecase:**
A document retrieval tool returns the summarized context in `.content` for the LLM and stores the full retrieved documents with metadata (source, page number, similarity score) in `.artifact` for citation and UI display.

**Common Pitfalls:**

* Storing large documents in `.content`, increasing token cost.
* Assuming `.artifact` is automatically visible to the LLM.
* Ignoring `.artifact`, making citations and debugging difficult.

**To Summarize:**
Remember: **`.content` is for the LLM, `.artifact` is for the application.** In RAG, keep the context in `.content` and the complete retrieval results in `.artifact`.


10. You're writing a `ChatPromptTemplate` whose system message needs to include a literal JSON
    example like `{"status": "ok"}`. What will go wrong if you paste that in directly, and how
    do you fix it?


**Answer:**
`ChatPromptTemplate` treats `{}` as **template variable placeholders**. If you paste a JSON object like `{"status": "ok"}` directly, LangChain will try to substitute `"status"` as a variable, resulting in a **missing variable (KeyError)** or template formatting error.

**Fix:** Escape the curly braces by doubling them.

```text
{{"status": "ok"}}
```

This tells `ChatPromptTemplate` to treat them as **literal braces**, not template variables.

**Usecase:**
Including JSON examples, API payloads, or configuration snippets inside system prompts without causing template parsing errors.

**Common Pitfalls:**

* Forgetting to escape `{}` in JSON examples.
* Mixing template variables with literal JSON.
* Escaping only one brace instead of both (`{{` and `}}`).

**To Summarize:**
Remember: **Single `{}` = template variable, Double `{{}}` = literal curly braces.**

11. What does `MessagesPlaceholder` do, and why can't you achieve the same result with a normal
    string-based template variable?
**Question:** What does `MessagesPlaceholder` do, and why can't you achieve the same result with a normal string-based template variable?

**Answer:**
`MessagesPlaceholder` is a special LangChain prompt component that inserts a **list of chat messages** (e.g., `SystemMessage`, `HumanMessage`, `AIMessage`, `ToolMessage`) into a chat prompt while preserving their roles and order.

A normal string template variable (`{chat_history}`) only inserts plain text, causing the conversation structure and message roles to be lost. This prevents the model from correctly interpreting multi-turn conversations and tool interactions.

**Usecase:**
Use `MessagesPlaceholder("chat_history")` in conversational agents to inject previous chat history or tool messages while maintaining the correct chat format.

**Common Pitfalls:**

* Using `{chat_history}` instead of `MessagesPlaceholder`, which flattens messages into plain text.
* Passing a string instead of a list of message objects.
* Losing tool-call and assistant message context by not preserving message roles.

**To Summarize:**
Remember: **`MessagesPlaceholder` preserves structured chat messages (roles + order), whereas a normal template variable only injects plain text.**


## A3. Structured Output (Medium-Hard)

12. Explain, in your own words, why asking a model to "please respond in JSON" via plain prompt
    instructions is fundamentally less reliable than using `with_structured_output()`.


**Answer:**
A plain prompt like *"Respond in JSON"* is only an **instruction**, so the model can still generate invalid JSON, miss required fields, add extra text, or change the schema.

`with_structured_output()` is more reliable because it **enforces a predefined schema** (typically using Pydantic or JSON Schema). The model is guided to produce structured data, and the output is validated before your application uses it, reducing parsing errors and improving consistency.

**Usecase:**
Extract structured information such as customer details, invoice fields, or product metadata into a validated Python object for downstream processing.

**Common Pitfalls:**

* Relying only on prompt instructions for JSON formatting.
* Not validating the model's output before using it.
* Assuming valid JSON always matches the expected schema.

**To Summarize:**
**Prompting asks** the model to follow a format; **`with_structured_output()` enforces** the format through schema validation, making it the preferred choice for production AI applications.


13. What is `model.profile`, and how is it actually used internally when you call
    `with_structured_output()` without specifying a strategy explicitly?


**Answer:**
`model.profile` is a **metadata object** that describes a model's capabilities (e.g., native structured output support, tool calling support, context window, etc.).

When you call `with_structured_output(schema)` **without explicitly choosing a strategy**, LangChain checks `model.profile` to automatically determine the best implementation:

* **If `model.profile` indicates native structured output support** → LangChain uses **ProviderStrategy**.
* **Otherwise** → LangChain falls back to **ToolStrategy**, where the schema is exposed as a tool/function call.

**Usecase:**
You can write:

```python
model.with_structured_output(MySchema)
```

without worrying about whether you're using OpenAI, Anthropic, Gemini, or another provider—LangChain automatically selects the appropriate structured output strategy based on the model's profile.

**Common Pitfalls:**

* Assuming `with_structured_output()` always uses native JSON mode—it depends on `model.profile`.
* Forgetting that if profile information is unavailable or incorrect, LangChain may choose a different strategy than expected.
* Not explicitly specifying `ProviderStrategy` or `ToolStrategy` when deterministic behavior is required across providers.

**To Summarize:**
**Remember:** *`model.profile` doesn't generate structured output—it advertises the model's capabilities. LangChain reads this metadata to automatically choose between **ProviderStrategy** and **ToolStrategy** when no strategy is explicitly provided.*


14. A teammate writes:
    `model.with_structured_output(BookingRequest, strategy=ProviderStrategy(BookingRequest))`
    and asks you to code review it. What's wrong with this line, and what should it be instead?


**Answer:**
The `strategy` parameter is **not supported** by `model.with_structured_output()`. The method automatically chooses the best structured output mechanism (provider-native or tool-calling) based on the model's capabilities.

**Correct code:**

```python
model.with_structured_output(BookingRequest)
```

If you want to explicitly use `ProviderStrategy`, it should be used with an **agent's `response_format`**, not with `with_structured_output()`.

**Usecase:**
Use `model.with_structured_output(BookingRequest)` when you want the LLM to always return a validated `BookingRequest` Pydantic object.

**Common Pitfalls:**

* Passing `strategy` to `with_structured_output()`.
* Confusing `with_structured_output()` with an agent's `response_format`.
* Assuming `ProviderStrategy` is required for every structured output.

**To Summarize:**
Remember:

* **`with_structured_output()` → Just pass the schema.**
* **`ProviderStrategy` / `ToolStrategy` → Used with agent `response_format`, not with `with_structured_output()`.**


15. What's the difference between structured output at the **model** level
    (`with_structured_output`) and at the **agent** level (`response_format` on `create_agent`)?
    Why does almost everything from Part 7 onward use the agent-level version?


**Answer:**

* **Model-level (`with_structured_output`)**: Applies structured output directly to a single LLM call. It ensures the model returns data in a predefined schema (e.g., Pydantic model or JSON schema). Best for standalone chains and simple workflows.

* **Agent-level (`response_format` in `create_agent`)**: Applies structured output to the **entire agent's final response**, even if the agent performs multiple reasoning steps and tool calls before answering.

**Why use the agent-level version from Part 7 onward?**
Because production agents are **tool-using, multi-step systems**. The agent-level `response_format` guarantees that the **final output remains structured**, regardless of how many tools or reasoning steps the agent executed internally.

**Usecase:**
A customer support agent uses search, database, and ticketing tools before returning a final response as:

```json
{
  "answer": "...",
  "confidence": 0.95,
  "citations": [...],
  "requires_human_support": false
}
```

**Common Pitfalls:**

* Using `with_structured_output()` for multi-step agents expecting the final agent response to be structured.
* Assuming every intermediate tool call follows the output schema.
* Mixing model-level and agent-level structured outputs unnecessarily.

**To Summarize:**
Remember this rule:

* **`with_structured_output()` → Structure one LLM call.**
* **`response_format` → Structure the final output of the entire agent.**

For production Agentic AI systems, prefer **agent-level structured output** because agents typically perform multiple tool calls before producing the final response.


16. You have a schema `response_format=ToolStrategy(Union[NewReservation, CancelReservation])`.
    Explain what happens internally when the model receives a message that's genuinely
    ambiguous between the two schemas.


**Answer:**
When using `ToolStrategy(Union[NewReservation, CancelReservation])`, the model analyzes the user's intent and selects the schema that best matches the request.

If the message is **genuinely ambiguous** (it could match both schemas), the model **does not randomly choose one**. Instead, it typically **asks a clarifying question** to gather more information before selecting the appropriate schema. This prevents incorrect structured outputs and ensures the response conforms to the correct schema.

**Usecase:**
User: *"I need help with my reservation."*
Since it's unclear whether the user wants to **create** or **cancel** a reservation, the model asks:
*"Would you like to make a new reservation or cancel an existing one?"*

**Common Pitfalls:**

* Assuming the model will always pick one schema automatically.
* Creating overlapping schemas that are difficult to distinguish.
* Ignoring clarification and forcing an incorrect structured output.

**To Summarize:**
**With `ToolStrategy(Union[...])`, the model first identifies the best-matching schema. If the intent is ambiguous, it asks a clarifying question instead of producing an invalid or incorrect structured response.**


17. A schema field is defined as `party_size: int = Field(ge=1, le=20)`, and a customer message
    says "table for 50 please." Walk through, step by step, what happens inside the agent loop
    from the moment the model first proposes `party_size=50` to the moment a valid final answer
    is produced.


**Answer:**

1. The model extracts `party_size=50` and proposes a tool call or structured output.
2. Pydantic validates the value against `ge=1` and `le=20`.
3. Validation fails because `50 > 20`.
4. The tool is not executed with the invalid value.
5. The validation error is returned to the agent loop as an observation.
6. The model re-evaluates the request and recognizes that the system supports a maximum party size of 20.
7. The model produces a valid response, such as asking the customer to split the booking or contact the restaurant for a large-group reservation.
8. The final answer is validated successfully and returned to the user.

**Usecase:**
Restaurant booking agents use schema validation to prevent invalid reservations from reaching the booking system.

**Common Pitfalls:**

* Silently changing `50` to `20` without user consent
* Executing the tool before validation
* Repeating the same invalid tool call indefinitely
* Exposing raw technical validation errors to the customer

**To Summarize:**
**Schema validation acts as a guardrail: the model may propose an invalid action, but the harness rejects it, feeds the error back, and lets the model recover with a valid response.**


## A4. Tools (Hard)

18. A tool's docstring is described as "the tool's entire pitch to the model," not documentation
    for humans. Defend or challenge this claim — is there ever a case where the docstring
    genuinely doesn't matter much?


**Answer:**
I **defend** this claim. For an LLM, the tool's docstring is the primary signal for **when to use the tool, why to use it, and how to use it correctly**. A clear, specific docstring improves tool selection and reduces incorrect or unnecessary tool calls.

The docstring matters **less** only in simple scenarios where:

* There is **only one available tool**.
* The tool name is **self-explanatory**.
* Tool selection is **hardcoded** by the application instead of being decided by the model.

In production systems with multiple tools, a well-written docstring is critical for reliable agent behavior.

**Usecase:**
A customer support agent with tools like `search_knowledge_base`, `create_ticket`, and `issue_refund` relies on clear docstrings so the model chooses the correct tool for each user request.

**Common Pitfalls:**

* Writing docstrings for developers instead of the LLM.
* Using vague descriptions like *"Utility function"*.
* Not specifying when the tool should or should not be used.
* Omitting input expectations or limitations.

**To Summarize:**
Remember: **A tool's docstring is prompt engineering for the model, not API documentation for humans. Better docstrings lead to better tool selection and more reliable agents.**


19. Explain what `ToolRuntime` actually hides from the model, and how LangChain knows to hide it
    (i.e., what specifically triggers this behavior)?


**Answer:**
`ToolRuntime` hides **runtime-only objects** from the LLM, such as the agent state, context, memory, configuration, store, stream writer, and tool execution metadata. These objects are injected **only at execution time** and are **not exposed in the tool schema** that the model sees.

LangChain knows to hide them because the tool function parameter is **type-annotated as `ToolRuntime`**. During tool schema generation, LangChain recognizes this special type and automatically excludes it from the arguments presented to the model while injecting it internally when the tool executes.

**Usecase:**
A customer support tool accesses the current conversation state or user context through `ToolRuntime` without requiring the LLM to generate those values.

**Common Pitfalls:**

* Assuming `ToolRuntime` is generated by the LLM.
* Expecting `ToolRuntime` fields to appear in the tool's input schema.
* Passing runtime information (e.g., state or memory) as normal tool arguments instead of using `ToolRuntime`.

**To Summarize:**
**Remember:** `ToolRuntime` is **hidden from the model** and **automatically injected by LangChain** based on its **type annotation**, keeping runtime context separate from LLM-generated tool inputs.


20. Compare `runtime.state`, `runtime.context`, and `runtime.store`. For each one, state: (a) how
    long the data persists, and (b) one concrete example of information that belongs there.


**Answer:**

| Component             | Persistence                                                                                            | Example                                                                                |
| --------------------- | ------------------------------------------------------------------------------------------------------ | -------------------------------------------------------------------------------------- |
| **`runtime.state`**   | Persists **only during the current agent run/workflow** (or until checkpointed in the same execution). | Current conversation messages, tool results, execution progress, intermediate outputs. |
| **`runtime.context`** | Exists **only for the current model invocation/request**. It is rebuilt for every LLM call.            | System prompt, retrieved documents, current user query, recent chat history.           |
| **`runtime.store`**   | Persists **across multiple runs and sessions**. It acts as long-term storage.                          | User preferences, profile information, long-term memory, previously saved documents.   |

**Usecase:**
A customer support agent stores the current conversation in **`runtime.state`**, sends the user's query and retrieved knowledge as **`runtime.context`**, and saves the user's preferred language in **`runtime.store`** for future conversations.

**Common Pitfalls:**

* Storing long-term user data in **`runtime.state`** instead of **`runtime.store`**.
* Assuming **`runtime.context`** persists across multiple LLM calls.
* Mixing temporary execution state with permanent application data.

**To Summarize:**
Remember the rule: **State = Current Workflow, Context = Current LLM Call, Store = Long-Term Memory.**


21. A tool accidentally declares a parameter named `config`. What actually happens when the
    agent tries to call it, and why is this a genuinely easy mistake to make by accident?

**Answer:**
If a tool declares a parameter named **`config`**, the agent runtime treats it as a **reserved/internal parameter** instead of a normal tool input. As a result, the model cannot populate it correctly, which can lead to tool invocation errors or unexpected behavior. This is an easy mistake because `config` is a very common parameter name developers naturally use for passing settings or configuration.

**Usecase:**
A developer writes:

```python
def search_docs(query: str, config: dict):
    ...
```

The agent may fail to call this tool correctly because `config` is interpreted as an internal runtime parameter.

**Common Pitfalls:**

* Using reserved parameter names like `config` for tool inputs.
* Mixing application configuration with user-provided tool arguments.
* Assuming every function parameter is exposed to the LLM.

**To Summarize:**
**Avoid using reserved runtime parameter names (such as `config`) in tool signatures.** Keep tool parameters limited to inputs the LLM should explicitly provide (e.g., `query`, `user_id`, `file_path`).


22. Explain the difference between a tool returning a plain string versus returning a `Command`.
    Give an original example (not from any notebook you've seen) of a situation that specifically
    requires `Command`, and explain why a plain string return wouldn't work for that case.


**Answer:**

* **Plain String Return:** Returns only data or a result (e.g., `"Order #123 has been shipped."`). It provides information but **does not change the agent's state or control the workflow**.
* **`Command` Return:** Returns both the result **and instructions** for the agent runtime, such as updating state, navigating to another node, interrupting execution, or routing the workflow.

A `Command` is required when a tool needs to **control the agent's execution**, not just provide information.

**Usecase:**
A **loan approval agent** calls a credit-check tool. If the credit score is below the threshold, the tool returns a `Command` to:

* Update the workflow state (`loan_status = "manual_review"`).
* Route execution to the **Human Review** node.

A plain string like `"Credit score is low"` cannot update the workflow state or change the execution path, so the agent would not know to pause and transfer the request for manual review.

**Common Pitfalls:**

* Using `Command` when only data needs to be returned.
* Returning a plain string when the tool must update state or control the workflow.
* Confusing **tool output** with **workflow control**.

**To Summarize:**
**Remember:** A **plain string communicates information**, while a **`Command` communicates both information and what the agent should do next (state update + workflow control).**


23. Describe, precisely, why `wrap_model_call`-based tool gating (making a tool invisible) is a
    stronger guarantee than instructing the model in the system prompt not to use a tool.
**Question:** Describe, precisely, why `wrap_model_call`-based tool gating (making a tool invisible) is a stronger guarantee than instructing the model in the system prompt not to use a tool.

**Answer:**
`wrap_model_call`-based tool gating is stronger because it **removes the tool from the model's available tool list**, making it impossible for the model to call it. A system prompt only **advises** the model not to use a tool, but the model can still ignore or misinterpret those instructions. In short, **tool gating enforces restrictions at the runtime level, while prompts rely on the model's compliance**.

**Usecase:**
Hide the **`issue_refund`** tool from regular support agents and expose it only after manager approval, ensuring the model cannot invoke refunds prematurely.

**Common Pitfalls:**

* Relying solely on system prompts for security.
* Assuming the model will always follow "do not use this tool" instructions.
* Exposing sensitive tools even when they should be inaccessible.

**To Summarize:**
**Prompt-based restrictions are advisory; `wrap_model_call` tool gating is enforced.** In production, always prefer **runtime-enforced access control** over prompt-based instructions for sensitive tools.


24. What is a **headless tool**, and how is its execution model fundamentally different from
    every other tool pattern covered in this course? Name one realistic capability that could
    only be implemented this way.


**Answer:**
A **headless tool** is a tool that runs **autonomously in the background without direct user interaction or a UI**, typically triggered by events, schedules, or workflows.

Unlike regular tools that execute **only when an agent explicitly calls them during its reasoning loop**, a headless tool operates **independently and asynchronously**, performing tasks without waiting for an active agent conversation.

**Usecase:**
A **background document indexing service** that automatically ingests new files, generates embeddings, and updates the vector database whenever documents are added—without any user prompt.

**Common Pitfalls:**

* Assuming headless tools are invoked by the agent like normal tools.
* Using headless tools for interactive tasks instead of background automation.
* Not implementing proper monitoring, retries, and error handling for long-running jobs.

**To Summarize:**
**Key difference:** *Regular tools are **agent-triggered** during reasoning, whereas headless tools are **event-driven or scheduled** and execute independently of the agent loop.*


---
# Part B — Coding Exercises

All exercises use **GreenPlate**, a restaurant reservation and food ordering assistant. Each
question gives you a blank code cell to work in — write your solution there and run it to
confirm it works before moving on.

## B1. Easy

**B1.1 — A basic tool.** Write a `@tool`-decorated function called `check_table_availability`
that takes a `party_size: int` and a `time_slot: str`, and returns a string saying whether a
table is available (you can hardcode fake availability data, e.g. tables available for parties
of 2-6 at "7:00 PM" and "8:30 PM" only). Include a proper docstring. Print the tool's `.name`,
`.description`, and `.args` to confirm it's built correctly.


In [ ]:
# Your solution for B1.1
from langchain_core.tools import tool
from pydantic import Field, BaseModel

class greenPlateAvailability(BaseModel):
  party_size: int = Field(ge=1, le=20, description="Number of person")
  time_slot: str = Field(description="Time slot for reservation")

@tool("checkTableAvailability", description="To check reservation availability", args_schema=greenPlateAvailability)
def check_table_availability(party_size: int, time_slot: str) -> str:
  """Use this tool to check table availability """
  return f"Table is available for parties of 2-6 at '7:00 PM' and '8:30 PM' only"

# print(check_table_availability.name)
# print(check_table_availability.description)
print(check_table_availability.args)

{'party_size': {'description': 'Number of person', 'maximum': 20, 'minimum': 1, 'title': 'Party Size', 'type': 'integer'}, 'time_slot': {'description': 'Time slot for reservation', 'title': 'Time Slot', 'type': 'string'}}


**B1.2 — A reusable prompt template.** Build a `ChatPromptTemplate` that generates a short,
enthusiastic description of a dish, given `{dish_name}` and `{cuisine_type}` as variables. Run
it with at least two different dish/cuisine combinations and print both results.


In [ ]:
# Your solution for B1.2
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model


# create a reusable ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a food critic who writes short and enthusiastic dish descriptions."),
    ("human", "Describe the dish {dish_name} from {cuisine_type} cuisine in 2-3 exciting sentences.")
])
model = init_chat_model("openai:gpt-5-mini")

formatted_prompt1 = prompt.invoke({"dish_name": "Misal", "cuisine_type": "Indian"})

response1 = model.invoke(formatted_prompt1)
print(response1.content)
print('*' * 60)

formatted_prompt2 = prompt.invoke({"dish_name": "Pizza", "cuisine_type": "Italian"})

response2 = model.invoke(formatted_prompt2)
print(response2.content)


/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/chat_models.py:431: UserWarning: Unrecognized keys in model profile: ['reasoning_effort_levels']. This may indicate a version mismatch between langchain-core and your provider package. Consider upgrading langchain-core.
  _warn_unknown_profile_keys(self.profile)


Misal is a fiery Maharashtrian bowl of sprouted moth beans simmered in a heady, red-chili-spiced gravy (tarri) and crowned with crunchy farsan or sev, chopped onions, coriander and a cheeky squeeze of lemon. Every spoonful crackles with hot, tangy, umami-packed flavors and textures, perfect for mopping up with soft pav or bhakri.
************************************************************
Pizza is a blistered, chewy Neapolitan disc of charred thin crust crowned with bright San Marzano tomato, silk-melted mozzarella and a scatter of fresh basil—each bite a thrilling balance of tang, creaminess and smoky crunch. From humble Margherita purity to toppings-studded extravaganzas, pizza is Italy’s joyful, communal alchemy: rustic, urgent and utterly irresistible.


**B1.3 — A schema with a constrained field.** Define a Pydantic `BaseModel` called
`FoodOrder` with fields: `customer_name` (str), `dish_name` (str), `quantity` (int, must be
between 1 and 10), and `spice_level` (a `Literal` restricted to `"mild"`, `"medium"`, or
`"hot"`). Use `with_structured_output()` to extract a `FoodOrder` from this message: *"Hi, I'm
Karan, 2 butter chicken please, medium spice."* Print the result.

In [ ]:
# Your solution for B1.3
from pydantic import BaseModel
from typing import Literal
from langchain.chat_models import init_chat_model

class FoodOrder(BaseModel):
  customer_name: str = Field(description="Name of the customer")
  dish_name: str = Field(description="Name of the dish")
  quantity: int = Field(ge=1, le=10, description="Quantity of the Dish")
  spice_level: Literal["mild", "medium", "hot"] = Field(description="Spice level of the dish")

model = init_chat_model("openai:gpt-5-mini")
response = model.with_structured_output(FoodOrder).invoke("Hi, I'm Karan, 2 butter chicken please, medium spice.")
print(response)

/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/chat_models.py:431: UserWarning: Unrecognized keys in model profile: ['reasoning_effort_levels']. This may indicate a version mismatch between langchain-core and your provider package. Consider upgrading langchain-core.
  _warn_unknown_profile_keys(self.profile)


customer_name='Karan' dish_name='Butter Chicken' quantity=2 spice_level='medium'


## B2. Medium

**B2.1 — Two schemas, one agent.** A restaurant assistant needs to handle both new reservations
and cancellations. Define `NewReservation` (customer_name, party_size, time_slot) and
`CancelReservation` (customer_name, time_slot) as separate Pydantic models. Build a
`create_agent` with `response_format=ToolStrategy(Union[NewReservation, CancelReservation])`,
and test it with one clearly-a-reservation message and one clearly-a-cancellation message. Use
`isinstance()` to print which schema was chosen each time.


In [ ]:
# Your solution for B2.1
from pydantic import BaseModel, Field
from typing import Literal, Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import AIMessage

class NewReservation(BaseModel):
  customer_name: str = Field(description="Name of the customer")
  party_size: int = Field(ge=1, le=20, description="Number of person")
  time_slot: str = Field(description="Time slot for reservation")

class CancelReservation(BaseModel):
  customer_name: str = Field(description="Name of the customer")
  time_slot: str = Field(description="Time slot for reservation")

agent = create_agent(
    model="openai:gpt-5-mini",
    response_format=ToolStrategy(Union[NewReservation, CancelReservation])
)

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Hi, I'm Karan, 2 butter chicken please, medium spice."
            }
        ]
    }
)

for message in response["messages"]:
    if isinstance(message, AIMessage) and message.tool_calls:
        if message.tool_calls[0]['name'] == "NewReservation":
            print("NewReservation")
        elif message.tool_calls[0]['name'] == "CancelReservation":
            print("CancelReservation")
        else:
            print("Unknown")


/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/chat_models.py:431: UserWarning: Unrecognized keys in model profile: ['reasoning_effort_levels']. This may indicate a version mismatch between langchain-core and your provider package. Consider upgrading langchain-core.
  _warn_unknown_profile_keys(self.profile)


NewReservation


**B2.2 — A tool with `args_schema`.** Define a Pydantic input schema called `OrderInput` with
`dish_name` (str, with a description), `quantity` (int, `ge=1, le=10`, with a description), and
`delivery_or_pickup` (a `Literal["delivery", "pickup"]`, defaulting to `"pickup"`). Build a tool
called `place_order` using `args_schema=OrderInput`. Print `place_order.args` to confirm the
schema came through correctly, including the constraint and the default.


In [ ]:
# Your solution for B2.2
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.tools import tool

class OrderInput(BaseModel):
  dish_name: str = Field(description="Name of the dish")
  quantity: int = Field(ge=1, le=10, description="Quantity of the")
  delivery_or_pickup: Literal["delivery", "pickup"] = Field(default="pickup", description="Delivery or pickup")

@tool("PlaceOrder", description="Tool used to place the order", args_schema=OrderInput)
def place_order():
  """Place orders for the customers"""
  print(f"Customers order placed!!")

print(place_order.args)

{'dish_name': {'description': 'Name of the dish', 'title': 'Dish Name', 'type': 'string'}, 'quantity': {'description': 'Quantity of the', 'maximum': 10, 'minimum': 1, 'title': 'Quantity', 'type': 'integer'}, 'delivery_or_pickup': {'default': 'pickup', 'description': 'Delivery or pickup', 'enum': ['delivery', 'pickup'], 'title': 'Delivery Or Pickup', 'type': 'string'}}


**B2.3 — Deliberately trigger a validation failure and watch self-correction.** Using the
`FoodOrder` schema from B1.3, build a `create_agent` with
`response_format=ToolStrategy(FoodOrder)`. Send a message asking for 15 units of a dish (this
should violate your `quantity` constraint). Print the full `result["messages"]` trace and point
out, in a comment, exactly where the self-correction happens.


In [ ]:
# Your solution for B2.3
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

class FoodOrder(BaseModel):
  customer_name: str = Field(description="Name of the customer")
  dish_name: str = Field(description="Name of the dish")
  quantity: int = Field(ge=1, le=10, description="Quantity of the Dish")
  spice_level: Literal["mild", "medium", "hot"] = Field(description="Spice level of the dish")

agent_food_order = create_agent(
    model="openai:gpt-5-mini",
    response_format=ToolStrategy(
        FoodOrder,
        handle_errors=(
            "The structured output is invalid. "
            "Correct the values so they satisfy the schema constraints."
        ),
      )
)

res_food_order = agent_food_order.invoke({
        "messages": [
            {
                "role": "user",
                "content": "Hi, I'm Karan, 15 sweet corn cheese pizza please, medium spicey."
            }
        ]
    })

print(res_food_order["messages"])


[HumanMessage(content="Hi, I'm Karan, 15 sweet corn cheese pizza please, medium spicey.", additional_kwargs={}, response_metadata={}, id='87c2cd30-538c-485c-8b88-853e1baeba2c'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 607, 'prompt_tokens': 195, 'total_tokens': 802, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EAY5qauDiIBI92i45PVX8ImHTVev0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fe0ce-417a-77c0-b3e9-2214ad99e650-0', tool_calls=[{'name': 'FoodOrder', 'args': {'customer_name': 'Karan', 'dish_name': 'sweet corn cheese pizza', 'quantity': 10, 'spice_level': 'medium'},

[HumanMessage(content="Hi, I'm Karan, 15 sweet corn cheese pizza please, medium spicey.", additional_kwargs={}, response_metadata={}, id='4b21c2c8-a71a-40c3-98c7-ab10bb328116'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 236, 'prompt_tokens': 195, 'total_tokens': 431, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EAXNzeZlKURLC3YgQJkCbzRBYywjz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fe0a4-c58a-78c3-af5d-60462cbf6545-0', tool_calls=[{'name': 'FoodOrder', 'args': {'customer_name': 'Karan', 'dish_name': 'Sweet Corn Cheese Pizza', 'quantity': 10, 'spice_level': 'medium'}, 'id': 'call_j71HMICRxVWJZndWOZk2AZbP', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 195, 'output_tokens': 236, 'total_tokens': 431, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 192}}), ToolMessage(content="Returning structured response: customer_name='Karan' dish_name='Sweet Corn Cheese Pizza' quantity=10 spice_level='medium'", name='FoodOrder', id='cf9d67a0-08ff-4c14-b9cc-3b07b01521ed', tool_call_id='call_j71HMICRxVWJZndWOZk2AZbP')]


with ToolStrategy, handle_errors=True by default, so a schema-validation failure is converted into a ToolMessage containing the validation error and instructions to fix the mistake; the model is then invoked again.

One subtle point: asking for 15 when the schema allows only 10 does not guarantee that the model will simply change 15 to 10. A well-behaved model may instead recognize that the user request cannot be represented faithfully and respond differently depending on your prompt. If your assignment specifically wants to visibly demonstrate automatic correction, you can make the retry instruction explicit:

response_format=ToolStrategy(
    FoodOrder,
    handle_errors=(
        "The structured output is invalid. "
        "Correct the values so they satisfy the schema constraints."
    ),
)


But for demonstrating LangChain’s built-in behavior, plain:


response_format=ToolStrategy(FoodOrder)


is enough because error handling and retry are enabled by default.

## B3. Hard

**B3.1 — Long-term memory with `ToolRuntime`.** Build two tools: `save_dietary_preference`
(takes `customer_id`, `preference`, and `runtime: ToolRuntime`, saves to `runtime.store`) and
`recall_dietary_preference` (takes `customer_id` and `runtime: ToolRuntime`, reads it back).
Build an agent with these two tools and a real `store=` attached. Prove the memory survives
across two **separate** `.invoke()` calls — save a preference in the first call, and recall it
correctly in a second, independent call.


In [ ]:
# Your solution for B3.1
from langgraph.store.memory import InMemoryStore
from langchain_core.tools import tool
from langchain.tools import tool,ToolRuntime
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from typing import Any

loyalty_store = InMemoryStore()

@tool
def save_dietary_preference(customer_id: str, preference: str, runtime: ToolRuntime) -> str:
  """Saves customers Dietary preference"""
  runtime.store.put(
    (customer_id, "preferences"),     # namespace
    "dietary_preference",             # key
    {"value": preference}             # value
  )
  return f"Got it, I will remember, you like {preference}"

@tool
def recall_dietary_preference(customer_id: str, runtime: ToolRuntime) -> str:
  """Recall customers Dietary preference"""
  preference = runtime.store.get(
    (customer_id, "preferences"),
    "dietary_preference"
  )
  return f"You like {preference}"

my_memory_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[save_dietary_preference, recall_dietary_preference],
    store=loyalty_store
)

my_memory_agent.invoke({"messages": [("user", "Hi, I am customer Sandeep001, I like protine food, please remember that.")]})

/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/chat_models.py:431: UserWarning: Unrecognized keys in model profile: ['reasoning_effort_levels']. This may indicate a version mismatch between langchain-core and your provider package. Consider upgrading langchain-core.
  _warn_unknown_profile_keys(self.profile)


{'messages': [HumanMessage(content='Hi, I am customer Sandeep001, I like protine food, please remember that.', additional_kwargs={}, response_metadata={}, id='25a8b5f4-3112-4b34-9a2c-3809a47db44b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 176, 'total_tokens': 277, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EAzlIYaAenB63Mqg32RA3ErJbDaVJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fe725-2529-7921-94bd-2f70a62bfbaa-0', tool_calls=[{'name': 'save_dietary_preference', 'args': {'customer_id': 'Sandeep001', 'preference': 'protein food'}, 'id': 'call

In [ ]:
result = my_memory_agent.invoke({"messages": [("user", "What food do I usually like? I'm Sandeep001.")]})
print(result['messages'][-1].content)

You usually like "protein food" (saved under Sandeep001).

Examples of protein-rich foods: chicken, fish, eggs, Greek yogurt, tofu, lentils/beans, nuts, and lean beef. Want recipe ideas, meal plans, or to update this preference?


**B3.2 — Dynamic tool gating.** GreenPlate has a `book_private_dining_room` tool that should
only be available to customers with a "premium" membership tier. Using `wrap_model_call`, write
a middleware function that removes this tool from the model's visible toolset unless
`request.state.get("is_premium_member")` is `True`. Prove it works by running the SAME query
twice — once without the flag, once with it — and show the tool is genuinely unavailable in the
first case (not just "declined").


In [ ]:
# Your solution for B3.2
from langchain_core.tools import tool
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@tool("StandardDiningRoomBooking", description="For standard booking and non-premium members only")
def book_standard_dining_room(customer_name: str, no_of_seats: int, return_direct=True) -> str:
  """Book a standard dining room. No further questions."""
  return f"Standard dining room is booked with name - {customer_name}."

@tool("PrivateDiningRoomBooking", description="For private booking and premium members only")
def book_private_dining_room(customer_name: str, no_of_seats: int, return_direct=True) -> str:
  """Book a Private dining room, for premium membership only. No further questions."""
  return f"Private dining room is booked with name - {customer_name}."

@wrap_model_call
def gate_private_tools(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Only expose book_private_dining_room to premium members."""
    is_premium = request.state.get("is_premium_member", False)
    if not is_premium:
        allowed = [t for t in request.tools if t.name != "book_private_dining_room"]
        request = request.override(tools=allowed)
    return handler(request)

gated_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[book_standard_dining_room, book_private_dining_room],
    middleware=[gate_private_tools],
)

result_regular = gated_agent.invoke({"messages": [("user", "Book me a standard dining room for Sandeep for 2 people?")], "is_premium_member": False})
print("Regular member result:", result_regular["messages"][-1].content)

/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/chat_models.py:431: UserWarning: Unrecognized keys in model profile: ['reasoning_effort_levels']. This may indicate a version mismatch between langchain-core and your provider package. Consider upgrading langchain-core.
  _warn_unknown_profile_keys(self.profile)


Regular member result: Done — standard dining room booked for Sandeep for 2 people.


In [ ]:
result_vip = gated_agent.invoke(
    {"messages": [("user", "Book me a private dining room for Sandeep for 2 people?")], "is_premium_member": True}
)
print("VIP member result:", result_vip["messages"][-1].content)

VIP member result: Done — private dining room booked for Sandeep for 2 people.


**B3.3 — Combine structured output and tools in one agent.** Build a `create_agent` that has
BOTH a `check_table_availability`-style tool AND a `response_format=ReservationConfirmation`
schema (define this schema yourself — it should capture at minimum: customer_name, time_slot,
confirmed: bool). Send a request that requires the agent to actually call the tool to check
availability *before* it can correctly fill in `confirmed`. Print both `result["messages"]` and
`result["structured_response"]`, and explain in a comment why this required the agent-level
`response_format`, not the raw model-level `with_structured_output()`.


In [ ]:
# Your solution for B3.3
from langchain_core.tools import tool
from pydantic import Field, BaseModel
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langgraph.store.memory import InMemoryStore
from langchain.tools import ToolRuntime

class ReservationConfirmation(BaseModel):
  customer_name: str = Field(description="Name of the customer")
  party_size: int = Field(ge=1, le=20, description="Number of person")
  time_slot: str = Field(description="Reservation booking time slot")
  booking_confirmation: bool = Field(description="Booking confirmed or not True - Yes, False - No")

# @tool("checkTableAvailability", description="To check reservation availability", args_schema=ReservationConfirmation)
# def check_table_availability(customer_name: str, party_size: int, time_slot: str, booking_confirmation: bool) -> str:
def check_table_availability(party_size: int, time_slot: str) -> str:
  """Check whether GreenPlate has a table available for the requested time."""
  # Simulated external reservation system
  availability = {
      "7:00 PM": True,
      "8:00 PM": False,
      "9:00 PM": True,
  }

  is_available = availability.get(time_slot, False)

  if is_available:
      return (
          f"A table for {party_size} people IS available "
          f"at {time_slot}."
      )

  return (
      f"A table for {party_size} people is NOT available "
      f"at {time_slot}."
  )

greenPlateReservationAgent = create_agent(
    model="openai:gpt-5-mini",

    tools=[check_table_availability],

    response_format=ToolStrategy(ReservationConfirmation),

    system_prompt="""
    You are GreenPlate's reservation assistant.

    Rules:
    1. Never guess whether a table is available.
    2. You MUST call check_table_availability before deciding
      whether confirmed should be True or False.
    3. confirmed=True only when the availability tool says
      the requested table is available.
    4. Return the final result using ReservationConfirmation.
    """
)

result = greenPlateReservationAgent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Hi, I'm Karan. Please reserve a table "
                    "for 4 people at 8:00 PM."
                )
            }
        ]
    }
)

print(result["structured_response"])

# WHY AGENT-LEVEL response_format?
#
# The agent first needs to call check_table_availability,
# receive the ToolMessage, reason over that result, and only
# THEN construct the final ReservationConfirmation.
#
# response_format belongs to create_agent because it structures
# the FINAL output of the complete multi-step agent loop.
#
# model.with_structured_output() operates at the raw model-call level;
# by itself, it does not represent the full
# tool -> observation -> reasoning -> final structured-response
# lifecycle that create_agent manages.
#
# create_agent() is explicitly an agent graph that can call tools repeatedly
# until a stopping condition is reached,
# while agent-level response_format governs the
# structured result of that overall agent execution.

customer_name='Karan' party_size=4 time_slot='8:00 PM' booking_confirmation=False


---
# Part C — Capstone Challenge

Build a single, complete `create_agent` for GreenPlate that combines **at least five** of the
following in one working system (your choice which five, but justify your choices in a markdown
cell before your code):

- A structured schema for orders or reservations (with at least one real constraint, like a
  `Literal` or a numeric range)
- At least two custom tools
- Long-term memory via `ToolRuntime.store` (e.g. remembering a customer's dietary preferences or
  favorite dish across sessions)
- Short-term memory via a checkpointer and `thread_id` (e.g. remembering the customer's name
  within one conversation)
- Dynamic tool gating based on some condition (membership tier, time of day, order size — your
  choice)
- A `context_schema` carrying some per-run data a tool reads (e.g. `restaurant_location`)

**Requirements:**
1. A markdown cell explaining your design choices before the code.
2. The complete, runnable code.
3. At least two `.invoke()` calls that demonstrate the system actually working — not just that
   it builds without error.
4. A short markdown reflection (3-5 sentences) on ONE trade-off or limitation of your design —
   what would break, or what would you need to add, if this went to real production use.

This is intentionally open-ended. There is no single correct architecture — the goal is
demonstrating you can combine these pieces into something coherent, not matching a hidden answer
key.


In [27]:
# Your capstone solution
from pydantic import BaseModel, Field
from typing import Literal, Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import AIMessage

from langchain_core.tools import tool
from langchain.tools import ToolRuntime
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable
from dataclasses import dataclass
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain.agents.middleware import ToolCallLimitMiddleware

from rich import print

loyalty_store = InMemoryStore()

@dataclass
class customerContext:
    customer_id: str
    customer_location: str


################ Strctured Schemas ###########################
class greenPlateNewReservation(BaseModel):
  customer_name: str = Field(description="Name of the customer")
  party_size: int = Field(ge=1, le=20, description="Number of person")
  time_slot: str = Field(description="Time slot for reservation")

class greenPlateCancelReservation(BaseModel):
  customer_name: str = Field(description="Name of the customer")
  time_slot: str = Field(description="Time slot for reservation")

class greenPlateFoodOrder(BaseModel):
  customer_name: str = Field(description="Name of the customer")
  dish_name: str = Field(description="Name of the dish")
  quantity: int = Field(ge=1, le=10, description="Quantity of the Dish")
  spice_level: Literal["mild", "medium", "hot"] = Field(description="Spice level of the dish")

################ Strctured Schemas END ###########################

################ Custom Tools ###########################
@tool("StandardDiningRoomBooking", description="For standard booking and non-premium members only")
def book_standard_dining_room(customer_name: str, no_of_seats: int, return_direct=True) -> str:
  """Book a standard dining room. No further questions."""
  return f"Standard dining room is booked with name - {customer_name}."

@tool("PrivateDiningRoomBooking", description="For private booking and premium members only")
def book_private_dining_room(customer_name: str, no_of_seats: int, return_direct=True) -> str:
  """Book a Private dining room, for premium membership only. No further questions."""
  return f"Private dining room is booked with name - {customer_name}."

@tool
def save_dietary_preference(customer_id: str, preference: str, runtime: ToolRuntime) -> str:
  """Saves customers Dietary preference"""
  runtime.store.put(
    (customer_id, "preferences"),     # namespace
    "dietary_preference",             # key
    {"value": preference}             # value
  )
  return f"Got it, I will remember, you like {preference}"

@tool
def recall_dietary_preference(customer_id: str, runtime: ToolRuntime) -> str:
  """Recall customers Dietary preference"""
  preference = runtime.store.get(
    (customer_id, "preferences"),
    "dietary_preference"
  )
  return f"You like {preference}"

################ Custom Tools END ###########################

################ Dynamic Tool Gating - Premium member ###########################
@wrap_model_call
def gate_private_tools(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Only expose book_private_dining_room to premium members."""
    is_premium = request.state.get("is_premium_member", False)
    if not is_premium:
        allowed = [t for t in request.tools if t.name != "book_private_dining_room"]
        request = request.override(tools=allowed)
    return handler(request)

################ Dynamic Tool Gating - END ###########################

greenPlateAgent = create_agent(
    model="openai:gpt-5-mini",
    response_format=ToolStrategy(Union[greenPlateNewReservation, greenPlateCancelReservation, greenPlateFoodOrder]),
    tools=[book_standard_dining_room, book_private_dining_room, save_dietary_preference, recall_dietary_preference],
    # middleware=[gate_private_tools],
    store=loyalty_store,
    middleware=[
        gate_private_tools,
        ModelCallLimitMiddleware(
            thread_limit=10,
            run_limit=5,
            exit_behavior="end",
        ),
        # Global limit
        ToolCallLimitMiddleware(
            thread_limit=5,
            run_limit=5,
        ),
    ],
)

##### User Request1
response1 = greenPlateAgent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Hi, I'm Sandeep, please remember I am pure Vegiterian. I would like to order 2 medium spice butter paneer please."
            }
        ],
       "is_premium_member": False
    },
    config={
        "configurable": {
            "thread_id": "greenplate_thread_001"
        }
    },
    context=customerContext(
        customer_id="sandeep_001",
        customer_location="Mumbai"
    ),
)
print('#'*100)
print(response1)

response2 = greenPlateAgent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Hi, I'm Sandeep, would like book private dining room please."
            }
        ],
       "is_premium_member": False
    },
    config={
        "configurable": {
            "thread_id": "greenplate_thread_001"
        }
    },
    context=customerContext(
        customer_id="sandeep_001",
        customer_location="Mumbai"
    ),
)
print('#'*100)
print(response2)

# response3 = greenPlateAgent.invoke(
#     {
#         "messages": [
#             {
#                 "role": "user",
#                 "content": "Hi, I'm Sandeep, please remember I am pure Vegiterian. I would like book standard dining room. And I would like to order 2 medium spice butter paneer please."
#             }
#         ],
#          "is_premium_member": False
#     },
#     config={
#         "configurable": {
#             "thread_id": "greenplate_thread_001"
#         }
#     },
#     context=customerContext(
#         customer_id="sandeep_001",
#         customer_location="Mumbai"
#     ),
# )
# print('#'*100)
# print(response3)

response4 = greenPlateAgent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Hi, I'm Rahul, please remember I am Non Vegiterian. I would like book private dining room. And I would like to order 2 spicy butter chicken masala please."
            }
        ],
        "is_premium_member": False
    },
    config={
        "configurable": {
            "thread_id": "greenplate_thread_002"
        }
    },
    context=customerContext(
        customer_id="Rahul_001",
        customer_location="Delhi"
    ),
)
print('#'*100)
print(response4)

##### User Request2
# result_regular = greenPlateAgent.invoke({"messages": [("user", "Book me a standard dining room for Sandeep for 2 people?")], "is_premium_member": False})
# print("Regular member result:", result_regular["messages"][-1].content)

##### User Request3
# result_vip = greenPlateAgent.invoke(
#     {"messages": [("user", "Book me a private dining room for Sandeep for 2 people?")], "is_premium_member": True}
# )
# print("VIP member result:", result_vip["messages"][-1].content)

/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/chat_models.py:431: UserWarning: Unrecognized keys in model profile: ['reasoning_effort_levels']. This may indicate a version mismatch between langchain-core and your provider package. Consider upgrading langchain-core.
  _warn_unknown_profile_keys(self.profile)
/usr/local/lib/python3.12/dist-packages/pydantic/functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=customerContext(customer_...tomer_location='Mumbai'), input_type=customerContext])
  function=lambda v, h: h(v), schema=original_schema
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=customerContext(customer_...tomer_locat

####################################################################################################

{
    'messages': [
        HumanMessage(
            content="Hi, I'm Sandeep, please remember I am pure Vegiterian. I would like to order 2 medium spice 
butter paneer please.",
            additional_kwargs={},
            response_metadata={},
            id='6693012e-9a56-4ba8-80b3-4124937d85f9'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 536,
                    'prompt_tokens': 437,
                    'total_tokens': 973,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 448,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-ECOBa0ywxMRg7F3FcdwN8FsvSPGr5',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ffaf2-55b8-7870-b42e-2c5fa0c3f061-0',
            tool_calls=[
                {
                    'name': 'save_dietary_preference',
                    'args': {'customer_id': 'Sandeep', 'preference': 'Pure vegetarian'},
                    'id': 'call_oFvxTamI9AxG6osOCsEFjjmH',
                    'type': 'tool_call'
                },
                {
                    'name': 'greenPlateFoodOrder',
                    'args': {
                        'customer_name': 'Sandeep',
                        'dish_name': 'Butter Paneer',
                        'quantity': 2,
                        'spice_level': 'medium'
                    },
                    'id': 'call_nwPRFPXyJVa6psqlkh0GKBWF',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 437,
                'output_tokens': 536,
                'total_tokens': 973,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 448}
            }
        ),
        ToolMessage(
            content="Returning structured response: customer_name='Sandeep' dish_name='Butter Paneer' quantity=2 
spice_level='medium'",
            name='greenPlateFoodOrder',
            id='a863a3ea-fd90-437f-b522-8e3a11610f85',
            tool_call_id='call_nwPRFPXyJVa6psqlkh0GKBWF'
        ),
        ToolMessage(
            content='Got it, I will remember, you like Pure vegetarian',
            name='save_dietary_preference',
            id='55482ae3-253b-4e40-9d8c-4a617909abb2',
            tool_call_id='call_oFvxTamI9AxG6osOCsEFjjmH'
        )
    ],
    'structured_response': greenPlateFoodOrder(
        customer_name='Sandeep',
        dish_name='Butter Paneer',
        quantity=2,
        spice_level='medium'
    )
}

/usr/local/lib/python3.12/dist-packages/pydantic/functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=customerContext(customer_...tomer_location='Mumbai'), input_type=customerContext])
  function=lambda v, h: h(v), schema=original_schema
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=customerContext(customer_...tomer_location='Mumbai'), input_type=customerContext])
  return self.__pydantic_serializer__.to_python(
/usr/local/lib/python3.12/dist-packages/pydantic/functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context',

####################################################################################################

{
    'messages': [
        HumanMessage(
            content="Hi, I'm Sandeep, would like book private dining room please.",
            additional_kwargs={},
            response_metadata={},
            id='d5c472ee-9d0c-4c9e-9524-cc4ee0f94eda'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 296,
                    'prompt_tokens': 422,
                    'total_tokens': 718,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 256,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-ECOBhVfiME9fP6RgepNQhQV1D5IOX',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ffaf2-7519-7532-b491-fb3f2e21d95b-0',
            tool_calls=[
                {
                    'name': 'PrivateDiningRoomBooking',
                    'args': {'customer_name': 'Sandeep', 'no_of_seats': 4, 'return_direct': True},
                    'id': 'call_tizVKVmPvzf7tdlfz4k8PU3B',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 422,
                'output_tokens': 296,
                'total_tokens': 718,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 256}
            }
        ),
        ToolMessage(
            content='Private dining room is booked with name - Sandeep.',
            name='PrivateDiningRoomBooking',
            id='47741718-5c89-4e20-be25-ed2573acdd1c',
            tool_call_id='call_tizVKVmPvzf7tdlfz4k8PU3B'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 559,
                    'prompt_tokens': 481,
                    'total_tokens': 1040,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 512,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-ECOBmmQ6QLM1uxdvMeSNSpOM0pnNC',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ffaf2-86b8-7730-a971-cbd3868f7d6a-0',
            tool_calls=[
                {
                    'name': 'save_dietary_preference',
                    'args': {'customer_id': 'Sandeep', 'preference': 'none'},
                    'id': 'call_XBffspSiT21tOWUVDtishZPj',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 481,
                'output_tokens': 559,
                'total_tokens': 1040,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_detail

/usr/local/lib/python3.12/dist-packages/pydantic/functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=customerContext(customer_...stomer_location='Delhi'), input_type=customerContext])
  function=lambda v, h: h(v), schema=original_schema
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=customerContext(customer_...stomer_location='Delhi'), input_type=customerContext])
  return self.__pydantic_serializer__.to_python(


####################################################################################################

{
    'messages': [
        HumanMessage(
            content="Hi, I'm Rahul, please remember I am Non Vegiterian. I would like book private dining room. And
I would like to order 2 spicy butter chicken masala please.",
            additional_kwargs={},
            response_metadata={},
            id='ae6a5e00-fbc4-440b-852a-ce09aca09404'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 793,
                    'prompt_tokens': 444,
                    'total_tokens': 1237,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 704,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-ECOCFqO5pSsfrMhtW7aSrmKQk1g6v',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ffaf2-f8be-7f42-a021-a533fb8ef28a-0',
            tool_calls=[
                {
                    'name': 'save_dietary_preference',
                    'args': {'customer_id': 'Rahul', 'preference': 'Non Vegetarian'},
                    'id': 'call_moHOYc9HmwVkTc6h6leIBDKA',
                    'type': 'tool_call'
                },
                {
                    'name': 'greenPlateFoodOrder',
                    'args': {
                        'customer_name': 'Rahul',
                        'dish_name': 'Spicy Butter Chicken Masala',
                        'quantity': 2,
                        'spice_level': 'hot'
                    },
                    'id': 'call_iPaiedwPWdpNGXpQWOkY6vUZ',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 444,
                'output_tokens': 793,
                'total_tokens': 1237,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 704}
            }
        ),
        ToolMessage(
            content="Returning structured response: customer_name='Rahul' dish_name='Spicy Butter Chicken Masala' 
quantity=2 spice_level='hot'",
            name='greenPlateFoodOrder',
            id='9c90595b-26de-4f87-a3cb-78457c09877e',
            tool_call_id='call_iPaiedwPWdpNGXpQWOkY6vUZ'
        ),
        ToolMessage(
            content='Got it, I will remember, you like Non Vegetarian',
            name='save_dietary_preference',
            id='eaae3611-5655-4ee1-b765-1c8552780905',
            tool_call_id='call_moHOYc9HmwVkTc6h6leIBDKA'
        )
    ],
    'structured_response': greenPlateFoodOrder(
        customer_name='Rahul',
        dish_name='Spicy Butter Chicken Masala',
        quantity=2,
        spice_level='hot'
    )
}